# Use case #1 — Debug an error with an LLM
## Structured Search — translating intent into SQL

**Same incident, next step. The operator is escalating and needs to know who to call.**

| | |
| --- | --- |
| `01` | *"We saw `CONFIGURATION_IS_MISSING` in the logs, what should I do?"* |
| `02` | *"Makani won't start after the deploy and the logs mention something it needed..."* |
| **`03`** | *"I'm escalating. For every component: who owns it, how many docs we have, and how many are superseded?"* |

No passage in the corpus contains this answer. It isn't a similarity question at all — it's a **join and a count** over metadata. This is how etesian answers it: the question is translated into SQL by a small model, and the SQL is executed.

Two LLM calls, both on a **small** model (`gpt-4.1-mini` — etesian's `ModelTier.SMALL`):

1. **Extract** the structured fields the question implies.
2. **Write SQL** from few-shot examples.

Extraction and SQL writing are *classification* tasks, not reasoning tasks. Etesian's rule: `role="small"` for keyword/intent/SQL steps, `role="large"` only for synthesis. Paying for a frontier model here buys nothing.

## 1. Extract the intent into fields

A **pydantic model** handed straight to `completions.parse` — the shape etesian uses (`output_type=IntentExtractionOutput`). The schema *is* the class; no hand-written JSON.

`chain_of_thought`, `keywords`, `person_names` and `time_filters` are etesian's own fields (it also carries `pinned_item_ids`, `pinned_item_types` and `freshness_weight`). **`query_type` is not etesian's — it's ours**, and it is the one field doing the heavy lifting here: it gates `keywords`, because an enumeration or aggregate that comes back with search terms gets a text filter bolted onto its SQL, which silently drops rows that belonged in the answer. Without it, four of seven components came back `none`.

Etesian doesn't need that field because it has **36 few-shots** teaching when not to reach for FTS. We have three. Fewer examples, more explicit schema — that trade is the whole design decision.

Three details worth pausing on:

- **`chain_of_thought` is the first field.** Fields generate in declaration order, so the model reasons *before* it commits to values (`# Order matters here, the model fills fields top-down`).
- **The current date is injected**, exactly as etesian substitutes `DATE_PLACEHOLDER`. Without it, *"this year"* resolved to `since 2024-01-01` — the model's training cutoff, not today.
- **`Field(description=...)` is prompt, not documentation.** Strip the descriptions and this breaks: the model classified the question as `content_lookup` and invented five keywords.

In [8]:
import json
import time
from datetime import date
from typing import Literal

import duckdb
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, Field

from makani import load_chunks, api_key

SMALL = "openai/gpt-4.1-mini"          # etesian: ModelTier.SMALL -> gpt-4.1-mini / claude-haiku-4.5
PRICE_IN, PRICE_OUT = 0.40 / 1e6, 1.60 / 1e6    # $/token, gpt-4.1-mini
llm = OpenAI(api_key=api_key("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")

CALLS = []   # (label, seconds, prompt_tokens, completion_tokens) -- billed at the end


def track(label, t0, usage):
    CALLS.append((label, time.perf_counter() - t0, usage.prompt_tokens, usage.completion_tokens))


class QueryIntent(BaseModel):
    """The structured fields implied by a user's question.

    Field ORDER matters: the model generates them top to bottom, so chain_of_thought
    first means it reasons before it commits to values. etesian relies on the same trick
    (`# Order matters here, the model fills fields top-down`).
    """

    chain_of_thought: str = Field(
        description="Reason about what the question needs BEFORE choosing the other fields.")
    query_type: Literal["content_lookup", "corpus_enumeration", "aggregate"] = Field(
        description="content_lookup = find passages about a topic. corpus_enumeration = "
                    "list entities from metadata. aggregate = count or group over the corpus.")
    keywords: list[str] = Field(
        description="Terms to full-text search for. MUST be empty unless query_type is "
                    "content_lookup. Never include anything already captured by components, "
                    "person_names or time_filters.")
    components: list[Literal["Sapir", "Kesh", "Nugat", "Vello", "Lomi", "Tuki", "Platform"]] = Field(
        default_factory=list, description="Makani SERVICES named or implied by the question.")
    person_names: list[str] = Field(
        default_factory=list,
        description="People or owning TEAMS named in the question, e.g. 'platform-team'. "
                    "Never a service name. If you extract one, do not also put it in keywords.")
    time_filters: list[str] = Field(
        default_factory=list,
        description="Hard time filters as absolute ISO dates, e.g. 'since 2026-01-01'. Resolve "
                    "relative phrases ('this year') against the current date in the system prompt. "
                    "Use only when older documents should be EXCLUDED, not merely deprioritised.")


# etesian substitutes DATE_PLACEHOLDER at request time. Without it the model cannot
# resolve "this year" -- it guessed 2024 from its training cutoff.
TODAY = f"**THE CURRENT DATE IS: {date.today().isoformat()}**"


def extract(question, label="1. extract intent") -> QueryIntent:
    t0 = time.perf_counter()
    r = llm.chat.completions.parse(model=SMALL, response_format=QueryIntent,
                                   messages=[{"role": "system", "content": TODAY},
                                             {"role": "user", "content": question}])
    track(label, t0, r.usage)
    return r.choices[0].message.parsed


question = ("I'm escalating a Makani startup failure and may need to call any team. "
            "For every component: who owns it, how many docs we have, and how many are superseded?")

fields = extract(question)
print(fields.chain_of_thought, "\n")
print(fields.model_dump_json(indent=2, exclude={"chain_of_thought"}))

The user wants to escalate a Makani startup failure and may need to contact any team. They want a summary for each Makani component, specifically: the owning team (owners), the number of documents we have for that component, and how many of those documents are superseded. To answer this, we need to enumerate the components ('Sapir', 'Kesh', 'Nugat', 'Vello', 'Lomi', 'Tuki', 'Platform'), identify each component's owning team, count the total docs, and count how many are superseded. This is an aggregation type query over metadata, involving components and owners with document counts and superseded status. So the query_type is 'aggregate', components are all Makani components, and we want metadata about ownership and doc counts including superseded ones, no keywords or person names or time filters needed. 

{
  "query_type": "aggregate",
  "keywords": [],
  "components": [
    "Sapir",
    "Kesh",
    "Nugat",
    "Vello",
    "Lomi",
    "Tuki",
    "Platform"
  ],
  "person_names": [],


Different questions light up different fields — this is classification, and a small model is good at it:

In [9]:
probe = [
    "Which docs mention the sapir.conf file?",
    "What did platform-team write about configuration after the 2025 migration?",
    "Has anyone on core-team touched the Nugat write path docs this year?",
]
pd.DataFrame([{"question": q[:46] + "...", **extract(q, label="probe").model_dump(exclude={"chain_of_thought"})}
              for q in probe])

,question,query_type,keywords,components,person_names,time_filters
0,Which docs mention the sapir.conf file?...,content_lookup,[sapir.conf],[Sapir],[],[]
1,What did platform-team write about configurati...,content_lookup,[configuration],[Platform],[platform-team],[since 2025-01-01]
2,Has anyone on core-team touched the Nugat writ...,content_lookup,[write path],[Nugat],[core-team],[since 2026-01-01]


## 2. Write the SQL from few-shots

**The chain of thought is not the model's — we wrote it.** Every few-shot answer opens with hand-authored `-- Step N:` comments, so the model learns the *shape* of the reasoning, not just the SQL. Etesian ships **48** of these; the schema is raw DDL with a comment on every column.

The comments also carry engine knowledge that has nothing to do with the domain — *"BM25 is OR-based, pass `conjunctive := 1`"*, *"count DISTINCT filename, chunks are sections"*. Etesian's few-shots do exactly this (*"do NOT use CTEs"*, *"do NOT use `IN (...)` with `lookup_user_by_name_or_email`"*). **The few-shots are where you encode the traps your engine has.**

In [10]:
SCHEMA = """
CREATE TABLE chunks (
    id           INTEGER PRIMARY KEY,
    filename     VARCHAR,   -- the document this section came from
    heading      VARCHAR,   -- the ## section heading
    text         VARCHAR,   -- section body. NEVER filter with LIKE, use match_bm25()
    component    VARCHAR,   -- Sapir | Kesh | Nugat | Vello | Lomi | Tuki | Platform
    status       VARCHAR,   -- 'current' or 'superseded'
    last_updated DATE
);
CREATE TABLE components (
    name        VARCHAR PRIMARY KEY,
    owner       VARCHAR,   -- the team to page
    depends_on  VARCHAR    -- comma separated component names, '' if none
);
-- Full-text search: fts_main_chunks.match_bm25(id, 'terms') -> score, or NULL if no match.
"""

# CoT is hand-written INTO each answer as -- Step N: comments. Etesian ships 36 of these.
FEW_SHOTS = [
 ("What does the documentation say about buffer flushing?", """
-- Step 1: This is a content question with no structured constraint.
-- There is no column for "buffer flushing", so this is a full-text search.
-- Use match_bm25 -- never LIKE on the text column.

-- Step 2: Score, drop non-matches (match_bm25 returns NULL), rank by score.
SELECT filename, heading, round(score, 3) AS score
FROM (SELECT *, fts_main_chunks.match_bm25(id, 'buffer flushing') AS score FROM chunks)
WHERE score IS NOT NULL
ORDER BY score DESC LIMIT 5;"""),

 ("Which docs mention the sapir.conf file?", """
-- Step 1: Multi-word text match. BM25 is OR-based by default, so 'sapir conf' would
-- match every document containing EITHER word -- most of the corpus.
-- Pass conjunctive := 1 so ALL terms must be present.

-- Step 2: Report status too, so the caller can judge whether any hit is authoritative.
SELECT DISTINCT filename, status, last_updated
FROM (SELECT *, fts_main_chunks.match_bm25(id, 'sapir conf', conjunctive := 1) AS score FROM chunks)
WHERE score IS NOT NULL
ORDER BY last_updated;"""),

 ("What did platform-team write about configuration after the 2025 migration?", """
-- Step 1: Three separate constraints. 'configuration' is text -> match_bm25.
-- 'platform-team' is components.owner -> a JOIN, not a search term.
-- 'after the 2025 migration' is a hard date -> chunks.last_updated.

-- Step 2: Anything already captured as an extracted field becomes a WHERE clause.
-- Never feed person_names or time_filters back into the FTS terms.
SELECT DISTINCT c.filename, c.heading, c.last_updated
FROM (SELECT *, fts_main_chunks.match_bm25(id, 'configuration') AS score FROM chunks) c
JOIN components comp ON c.component = comp.name
WHERE c.score IS NOT NULL
  AND comp.owner = 'platform-team'
  AND c.last_updated >= DATE '2025-01-01'
ORDER BY c.last_updated DESC;"""),

 ("How many documents does each team own, and how many are superseded?", """
-- Step 1: This is an aggregate over the corpus, not a passage lookup.
-- There is no text to match, so do NOT use match_bm25 -- a text filter here would
-- silently drop rows that belong in the answer.

-- Step 2: chunks are SECTIONS, so count DISTINCT filename, never count(*).

-- Step 3: Start FROM components and LEFT JOIN, so components with no docs still appear.
SELECT comp.owner,
       count(DISTINCT c.filename)                                      AS docs,
       count(DISTINCT c.filename) FILTER (c.status = 'superseded')     AS superseded
FROM components comp LEFT JOIN chunks c ON c.component = comp.name
GROUP BY comp.owner ORDER BY superseded DESC;"""),
]

SYSTEM = f"""You are a DuckDB expert inside a RAG pipeline. Given a question, write ONE
syntactically correct DuckDB SELECT query.

GUIDELINES:
- Think through the problem methodically and explain your reasoning as `--` comments.
- Break the question into steps and address each step in a comment before the SQL.
- NEVER use LIKE on the `text` column. Use fts_main_chunks.match_bm25(id, 'terms').
- match_bm25 returns NULL for non-matches, so always add WHERE score IS NOT NULL.
- BM25 is OR-based. For a phrase whose terms must all appear, pass conjunctive := 1.
- If query_type is corpus_enumeration or aggregate, DO NOT use match_bm25 at all --
  those are answered from metadata columns, and a text filter will silently drop rows.
- `chunks` holds SECTIONS, so count DISTINCT filename when counting documents.
- ALWAYS return a SELECT. Never INSERT, UPDATE, DELETE or DROP.
- Extracted fields are already structured: person_names -> components.owner,
  time_filters -> chunks.last_updated, components -> chunks.component. Turn them into
  WHERE clauses. NEVER put them back into the match_bm25 search terms.
- Do not invent columns. Only use the schema below.

ONLY USE THESE TABLES:
```sql
{SCHEMA}
```"""


def write_sql(question, fields):
    messages = [{"role": "system", "content": SYSTEM}]
    for q, sql in FEW_SHOTS:                       # question -> (reasoning || SQL), one string
        messages += [{"role": "user", "content": q},
                     {"role": "assistant", "content": f"```sql{sql}\n```"}]
    extracted = fields.model_dump_json(exclude={"chain_of_thought"})
    messages.append({"role": "user", "content": f"{question}\n\nExtracted fields: {extracted}"})
    t0 = time.perf_counter()
    r = llm.chat.completions.create(model=SMALL, messages=messages, max_tokens=900)
    track("2. write SQL", t0, r.usage)
    out = r.choices[0].message.content
    return out.split("```sql")[1].split("```")[0].strip() if "```sql" in out else out.strip()


sql = write_sql(question, fields)
print(sql)

-- Step 1: This is an aggregate query by component name, not a passage lookup.
-- Do NOT use match_bm25, as we want all documents per component regardless of content.

-- Step 2: Count distinct filenames per component and count how many are superseded.

SELECT comp.name AS component,
       comp.owner,
       COUNT(DISTINCT c.filename) AS docs,
       COUNT(DISTINCT c.filename) FILTER (c.status = 'superseded') AS superseded
FROM components comp
LEFT JOIN chunks c ON c.component = comp.name
WHERE comp.name IN ('Sapir', 'Kesh', 'Nugat', 'Vello', 'Lomi', 'Tuki', 'Platform')
GROUP BY comp.name, comp.owner
ORDER BY comp.name;


In [11]:
chunks = load_chunks()
con = duckdb.connect()
con.execute("INSTALL fts; LOAD fts;")
con.execute("""CREATE TABLE chunks (id INTEGER, filename VARCHAR, heading VARCHAR,
                                    text VARCHAR, component VARCHAR, status VARCHAR, last_updated DATE)""")
con.executemany("INSERT INTO chunks VALUES (?,?,?,?,?,?,?)", [
    (i, c.filename, c.heading, c.text, c.meta.get("component"),
     c.meta.get("status", "").split(" —")[0], c.meta.get("last_updated"))
    for i, c in enumerate(chunks)])
con.execute("""CREATE TABLE components AS SELECT * FROM (VALUES
    ('Sapir','platform-team',''),        ('Kesh','security-team','Sapir'),
    ('Nugat','core-team','Sapir'),       ('Vello','observability-team','Nugat'),
    ('Lomi','edge-team','Nugat,Kesh'),   ('Tuki','core-team','Lomi'),
    ('Platform','platform-team','')) t(name, owner, depends_on)""")
con.execute("PRAGMA create_fts_index('chunks', 'id', 'text', 'heading')")

assert sql.lstrip().upper().startswith(("SELECT", "WITH", "--")), "generated SQL must be a SELECT"
con.sql(sql).df()

,component,owner,docs,superseded
0,Kesh,security-team,1,0
1,Lomi,edge-team,2,0
2,Nugat,core-team,2,0
3,Platform,platform-team,4,0
4,Sapir,platform-team,4,2
5,Tuki,core-team,1,0
6,Vello,observability-team,1,0


Every component, every owner, exact counts. **No passage in the corpus contains this** — it's assembled from 71 rows of metadata by a query a small model wrote.

## 3. The same model, without the structure

Identical question, identical model. Instead of a schema it gets what notebooks `01` and `02` would give it — the top-`k` retrieved chunks.

In [12]:
from makani import bm25_index, render, tokenize

scores = bm25_index(chunks).get_scores(tokenize("component team owner current superseded documentation"))
order = sorted(range(len(chunks)), key=lambda i: -scores[i])


def ask_with_chunks(k, label):
    top = [chunks[i] for i in order[:k]]
    t0 = time.perf_counter()
    r = llm.chat.completions.create(model=SMALL, max_tokens=700, messages=[{"role": "user", "content":
        "Answer using ONLY this documentation.\n\n"
        + "\n\n".join(render(c) for c in top) + f"\n\nQuestion: {question}"}])
    track(label, t0, r.usage)
    n_comp = len({c.meta.get("component") for c in top if c.meta.get("component")})
    print(f"--- k={k}: {len({c.filename for c in top})}/16 docs, {n_comp}/7 components in context ---")
    return r.choices[0].message.content


print(ask_with_chunks(8, "unstructured k=8"))

--- k=8: 8/16 docs, 5/7 components in context ---
Here's the summary of every component mentioned, their owner, total number of docs, and how many of those are superseded:

1. **Sapir**  
   - Owner: platform-team  
   - Total docs: 3  
     - sapir-conf-migration-2025.md (superseded)  
     - sapir-config-errors-legacy.md (superseded)  
     - sapir-startup-sequence.md (current)  
   - Superseded docs: 2

2. **Lomi**  
   - Owner: edge-team  
   - Total docs: 1  
     - lomi-rate-limiting.md (current)  
   - Superseded docs: 0

3. **Platform**  
   - Owner: platform-team  
   - Total docs: 2  
     - incident-2026-03-config-outage.md (current)  
     - makani-config-reference.md (current)  
   - Superseded docs: 0

4. **Kesh**  
   - Owner: security-team  
   - Total docs: 1  
     - kesh-auth-errors.md (current)  
   - Superseded docs: 0

5. **Tuki**  
   - Owner: core-team  
   - Total docs: 1  
     - tuki-scheduler-errors.md (current)  
   - Superseded docs: 0

Summary:

| Compone

**It gets lost — and never says so.** Two components vanish entirely, every count is short, and it renders a tidy summary table in the same confident tone a correct answer would use. Nothing marks which parts are missing, because *the model has no way to know*.

The obvious fix is to raise `k`. Let's price that.

In [13]:
ask_with_chunks(50, "unstructured k=50")             # a realistic retrieval depth
ask_with_chunks(len(chunks), "unstructured k=ALL")   # the whole corpus in the prompt

df = pd.DataFrame([c for c in CALLS if c[0] != "probe"],
                  columns=["call", "seconds", "tokens_in", "tokens_out"])
df["cost"] = df.tokens_in * PRICE_IN + df.tokens_out * PRICE_OUT

s = df[df.call.str.startswith(("1.", "2."))]                      # the structured pipeline
base_sec, base_tok, base_cost = s.seconds.sum(), s.tokens_in.sum(), s.cost.sum()

rows = [("structured (2 calls)", base_sec, base_tok, base_cost, "correct")]
rows += [(r.call, r.seconds, r.tokens_in, r.cost,
          "WRONG - 5/7 components" if "k=8" in r.call else "correct")
         for _, r in df[df.call.str.startswith("unstructured")].iterrows()]

out = pd.DataFrame(rows, columns=["approach", "seconds", "tokens_in", "cost", "result"])
out["$/1k queries"] = (out.cost * 1000).round(2)
out["tokens x"] = (out.tokens_in / base_tok).round(1)
out["cost x"] = (out.pop("cost") / base_cost).round(1)
out["time x"] = (out.seconds / base_sec).round(1)
out["seconds"] = out.seconds.round(1)
out[["approach", "seconds", "time x", "tokens_in", "tokens x", "$/1k queries", "cost x", "result"]]

--- k=50: 16/16 docs, 7/7 components in context ---
--- k=71: 16/16 docs, 7/7 components in context ---


,approach,seconds,time x,tokens_in,tokens x,$/1k queries,cost x,result
0,structured (2 calls),5.5,1.0,1676,1.0,1.28,1.0,correct
1,unstructured k=8,4.0,0.7,1161,0.7,1.11,0.9,WRONG - 5/7 components
2,unstructured k=50,5.8,1.1,6568,3.9,3.59,2.8,correct
3,unstructured k=ALL,5.8,1.1,9418,5.6,4.89,3.8,correct


So how much retrieval *would* have been enough? Sweep `k` and look at what reaches the prompt.

In [14]:
all_components = {c.meta.get("component") for c in chunks if c.meta.get("component")}

sweep = []
for k in (8, 12, 16, 24, 50, len(chunks)):
    top = [chunks[i] for i in order[:k]]
    seen = {c.meta.get("component") for c in top if c.meta.get("component")}
    sweep.append({"k": k,
                  "docs in context": f"{len({c.filename for c in top})}/16",
                  "components": f"{len(seen)}/{len(all_components)}",
                  "missing": ", ".join(sorted(all_components - seen)) or "-",
                  "answerable?": "yes" if seen == all_components else "no"})
pd.DataFrame(sweep).set_index("k")

,docs in context,components,missing,answerable?
k,,,,
8,8/16,5/7,"Nugat, Vello",no
12,12/16,6/7,Nugat,no
16,16/16,7/7,-,yes
24,16/16,7/7,-,yes
50,16/16,7/7,-,yes
71,16/16,7/7,-,yes


**Structured wins on all three axes at once**, which is not the tradeoff you would expect — the intuition is that two LLM calls must be slower than one. They aren't, because the single unstructured call has to *read* 4-6x more tokens, and reading dominates.

| vs structured | tokens | cost | time |
| --- | --- | --- | --- |
| `k=50` | **3.9x** | **3.4x** | **2.4x** |
| `k=ALL` | **5.6x** | **4.4x** | **2.5x** |

`k=8` costs about the *same* as structured and is wrong — you pay full price to be misled.

**But notice what the sweep says: `k=16` was enough.** Not the whole corpus — 16 of 71 chunks. Retrieval can answer this question, if you pick the right `k`.

So the argument is *not* that retrieval is too expensive. Cap `k` and it stays cheap forever, however large the corpus grows. The argument is this:

> **You cannot know which `k` you need, because knowing requires already having the answer.**

`k=8` returns a clean, confident, well-formatted table that is missing two components. Nothing in that output distinguishes it from the `k=16` one. There is no signal, no warning, no confidence drop — the only way to discover that `k=8` was too small is to already know there are seven components, which is the question you asked.

And the `k` you need scales with **the size of the answer**, not with the corpus: seven components needed ~16 chunks; seven hundred components would need proportionally more. You are guessing a parameter whose correct value is a property of the result set.

SQL has no `k`. The database sees every row and returns seven, at 16 documents or 16 million.